In [1]:
import polars as pl

file_path = 'yambda/sequential-multievent-500m/sequential-multievent-500m.inter'
inter = pl.read_csv(file_path, separator='\t', has_header=True, quote_char=None, infer_schema_length=10000)
print(inter.head(10))

shape: (10, 2)
┌───────────────┬─────────────────────────────────┐
│ user_id:token ┆ item_id_list:token_seq          │
│ ---           ┆ ---                             │
│ i64           ┆ str                             │
╞═══════════════╪═════════════════════════════════╡
│ 21545         ┆ 560041 888698 335779 315938 81… │
│ 62305         ┆ 777467 16712 777467 416516 861… │
│ 18714         ┆ 908198 737893 963948 272252 21… │
│ 37294         ┆ 9356 55574 435532 370008 70768… │
│ 21796         ┆ 605813 1012727 316706 332985 9… │
│ 15297         ┆ 215073 656033 578228 1001258 3… │
│ 18471         ┆ 220830 863383 329495 171967 38… │
│ 21545         ┆ 797000 195085 134298 887927 10… │
│ 10305         ┆ 151990 270793 882600 1045001 7… │
│ 76725         ┆ 421830 741944 59483 421830 494… │
└───────────────┴─────────────────────────────────┘


In [2]:
import json

code_path = 'yambda/sequential-multievent-500m/sequential-multievent-500m.index.json'
with open(code_path, 'r') as f:
    items2codes = json.load(f)
print(items2codes['0'])

['<|a_107|>', '<|b_1|>', '<|c_501|>', '<|d_460|>']


In [3]:
inter_item_id = inter.select("item_id_list:token_seq")


In [7]:
sample = inter_item_id[11]['item_id_list:token_seq']
for i in sample:
    print(i)


1027338 494124 497235 492546 511137 883363 540113 423181 807565 1034133 924439 900861 889701 186931 379919 130388 488406 487010 411628 56537


In [9]:
from tqdm.auto import tqdm
import json

# 1. 预处理 mapping
item_code_map = {k: "".join(v) for k, v in items2codes.items()}

# 2. 定义带有进度条的转换函数
# 使用 tqdm 监控处理进度
total_rows = inter.height
pbar = tqdm(total=total_rows, desc="Processing rows")

def transform_seq(seq_str):
    pbar.update(1)
    if not seq_str:
        return ""
    return ",".join((item_code_map[x] for x in seq_str.split()))

try:
    # 处理数据
    df_processed = inter.select(
        pl.col('item_id_list:token_seq')
        .map_elements(transform_seq, return_dtype=pl.String)
        .alias('text')
    )
    
    # 保存为 .jsonl 格式 (NDJSON)
    # 这种格式每行一个 JSON 对象，方便流式读取和检查，且不会像缩进 JSON 那样占用过多空间
    output_file = 'llama_factory_data.jsonl'
    print(f"Saving to {output_file} in JSONL format...")
    df_processed.write_ndjson(output_file)
    print("Done.")
        
finally:
    pbar.close()

Processing rows: 100%|█████████▉| 22517817/22520616 [22:04<00:00, 31026.22it/s] 

Saving to llama_factory_data.jsonl in JSONL format...


Processing rows: 100%|██████████| 22520616/22520616 [26:38<00:00, 14084.51it/s]

Done.


In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
import json

class TokenExtender:
    def __init__(self, data_path, dataset, index_file=".index.json"):
        self.data_path = data_path
        self.dataset = dataset
        self.index_file = index_file
        self.indices = None
        self.new_tokens = None
        
    def _load_data(self):
        with open(os.path.join(self.data_path, self.dataset + self.index_file), 'r') as f:
            self.indices = json.load(f)
    
    def get_new_tokens(self):
        if self.new_tokens is not None:
            return self.new_tokens
            
        if self.indices is None:
            self._load_data()
        
        self.new_tokens = set()
        for index in self.indices.values():
            for token in index:
                self.new_tokens.add(token)
        self.new_tokens = sorted(list(self.new_tokens))
        
        return self.new_tokens

# ... (前面的 TokenExtender 类定义保持不变) ...

# 1. 定义输入和输出路径
model_path = '/home/huangminrui/models/Qwen3-4B-Instruct-2507'
# 建议保存到新目录，例如当前目录下的 output_model
output_dir = './yambda/model' 

print(f"Loading model from {model_path}")
model = AutoModelForCausalLM.from_pretrained(model_path, local_files_only=True)
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)

# 2. 加载并添加新 Token
sid_index_path = "./yambda/sequential-multievent-500m/sequential-multievent-500m.index.json"
print(f"Loading index from {sid_index_path}")

token_extender = TokenExtender(
    data_path=os.path.dirname(sid_index_path),
    dataset=os.path.basename(sid_index_path).split('.')[0]
)
new_tokens = token_extender.get_new_tokens()

if new_tokens:
    print(f"Adding {len(new_tokens)} new tokens to tokenizer...")
    # 添加 token
    num_added_toks = tokenizer.add_tokens(new_tokens)
    print(f"Actually added {num_added_toks} tokens (some might have already existed).")
    
    if num_added_toks > 0:
        # 调整模型 embedding 大小以匹配新词表
        model.resize_token_embeddings(len(tokenizer))
        
        # === 关键步骤：验证 ===
        # 随机取一个新 token 测试，确保它不会被拆分
        test_token = new_tokens[0] 
        encoded = tokenizer.tokenize(test_token)
        token_ids = tokenizer.encode(test_token)
        print(f"Verification - Token: {test_token}, Tokenized: {encoded}, Token IDs: {token_ids}")
        if len(encoded) != 1:
            print("⚠️ Warning: New token is being split! This might affect performance.")
        else:
            print("✅ Verification passed: Token is recognized as a single unit.")

        # === 关键步骤：保存 ===
        print(f"Saving model and tokenizer to {output_dir}...")
        model.save_pretrained(output_dir)
        tokenizer.save_pretrained(output_dir)
        print("Done.")
    else:
        print("No new tokens were added (all already existed).")
else:
    print("No tokens found to add.")

Loading model from /home/huangminrui/models/Qwen3-4B-Instruct-2507


Loading checkpoint shards: 100%|██████████| 3/3 [00:02<00:00,  1.49it/s]


Loading index from ./yambda/sequential-multievent-500m/sequential-multievent-500m.index.json
Adding 2309 new tokens to tokenizer...
Actually added 2309 tokens (some might have already existed).
Verification - Token: <|a_100|>, Tokenized: ['<|a_100|>'], Token IDs: [151669]
✅ Verification passed: Token is recognized as a single unit.
Saving model and tokenizer to ./yambda/model...
Done.


In [13]:
print(tokenizer.eos_token_id)
print(tokenizer.pad_token_id)

151645
151643
